# Task 2 and Task 3 Analysis Evidence - dabi0142


## Member Scope
This notebook contains the workflow and analysis completed by `dabi0142` for Task 2 and Task 3 of the assignment. The main purpose of the notebook is to collect POI data for the selected SA4 region, process the spatial datasets, calculate well-resourced scores for each SA2 area, and generate visualisations for analysis.

The notebook includes database queries, score calculations, spatial visualisations, and comparisons between different SA2 regions. Several charts and maps were produced to examine how resources and accessibility are distributed across Greater Sydney, with a particular focus on the Sydney - Parramatta SA4 region selected by dabi0142.


In [1]:
from dataclasses import replace

import pandas as pd

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table
from data2001.task4.maps import build_score_choropleth_map, build_poi_density_choropleth_map, build_poi_point_scatter_map

from data2001.task4.queries import (
    load_api_extraction_summary,
    load_correlation_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
)


MEMBER_UNIKEY = "dabi0142"

base_settings = load_settings("configs/local.yaml")
member_sa4 = (base_settings.task2_import.selected_sa4_by_member.get(MEMBER_UNIKEY) or "").strip()
if not member_sa4:
    raise ValueError(f"No SA4 configured for {MEMBER_UNIKEY}. Fill configs/local.yaml selected_sa4_by_member.")

settings = replace(
    base_settings,
    task2_import=replace(
        base_settings.task2_import,
        crawl_scope="selected_sa4",
        selected_sa4_by_member={MEMBER_UNIKEY: member_sa4},
    ),
    task3_score=replace(base_settings.task3_score, score_universe="selected_sa4"),
)
engine = create_engine_from_settings(settings.database)

member_scope = pd.DataFrame([{"unikey": MEMBER_UNIKEY, "selected_sa4": member_sa4}])
display(member_scope)

,unikey,selected_sa4
0,dabi0142,Sydney - Parramatta


## Single-SA4 Full Workflow Run
This section runs the complete workflow for the selected SA4 region. The workflow includes database initialisation, clearing previous tables, importing SA2 boundary data, importing POI and income datasets, and calculating the final well-resourced scores for each SA2 area.

The workflow summary output is used as evidence that the processing pipeline was executed successfully before generating the final analysis and visualisations.

In [2]:
workflow_steps = [
    "init_db",
    "clear_db",
    "import_boundaries",
    "validate_boundaries",
    "import_poi",
    "import_income",
    "compute_score",
]

workflow_summary = execute_workflow_steps(
    engine,
    settings,
    workflow_steps,
    title=f"{MEMBER_UNIKEY} single-SA4 full rebuild",
)
display(workflow_summary)

{'init_db': 'done',
 'clear_db': 'done',
 'sa4': 1,
 'sa2': 34,
 'population': 2473,
 'boundary_selected_sa4': 1,
 'boundary_sa2_checked': 34,
 'boundary_sa2_valid': 34,
 'boundary_sa2_invalid': 0,
 'boundary_min_coverage_ratio': 0.999999999999997,
 'boundary_coverage_threshold': 0.999,
 'boundary_point_failures': 0,
 'boundary_missing_parent_sa4': 0,
 'sa2_bbox_requests': 34,
 'raw_responses': 34,
 'raw_features_seen': 4085,
 'clean_features_seen': 2365,
 'fetch_seconds': 4.577883503065095,
 'persist_seconds': 0.04904700996121392,
 'clean_seconds': 0.14705981699808035,
 'load_seconds': 0.03889479399367701,
 'income': 2454,
 'scores': 31,
 'correlations': 2}

## Single-SA4 Database Verification
This section verifies that the database import process completed successfully for the selected SA4 region. The query checks the number of SA2 regions stored in the database after the workflow execution.

The output confirms that the Sydney - Parramatta SA4 region was imported correctly and contains 34 SA2 regions available for further analysis and score calculations.


In [3]:
schema = settings.database.schema_name

display(pd.read_sql(
    f"""
    SELECT sa4_name, COUNT(*) AS sa2_count
    FROM {schema}.sa2
    GROUP BY sa4_name
    """,
    engine,
))

,sa4_name,sa2_count
0,Sydney - Parramatta,34


## Task 2 Evidence: API Extraction and Spatial Join
This section provides evidence that the NSW Points of Interest API extraction and spatial join process were completed successfully.

The API extraction summary shows that response files were collected for all selected SA2 regions and converted into a combined POI dataset. After the extraction process, the spatial join workflow was used to match POI locations with their corresponding SA2 boundaries.

The output also shows the number of cleaned POIs, assigned POIs, and unassigned records after the spatial join process. Most POIs were successfully matched to SA2 regions, indicating that the boundary matching and coordinate processing steps worked correctly before the scoring analysis was performed.


In [4]:
display(load_api_extraction_summary(settings))
display(load_spatial_join_summary(engine, settings))

,response_dir,response_file_count,features_jsonl,raw_feature_rows,features_file_exists,features_file_size_mb
0,/home/kscii/Codes/data2001-group-assignment/da...,34,/home/kscii/Codes/data2001-group-assignment/da...,4085,True,2.15


,clean_poi,assigned_poi,unassigned_poi,boundary_duplicate_candidates,assignment_rows
0,2365,2086,279,0,2086


## Task 3 Evidence: Score Calculation
This section demonstrates how the well-resourced scores were calculated for each SA2 region within the selected SA4 area.

The summary statistics show the total number of SA2 regions included in the analysis, the total number of POIs collected, and the distribution of POI counts across the dataset. The scoring workflow used standardised z-scores and a sigmoid transformation to convert POI counts into comparable well-resourced scores between 0 and 100.

The results show noticeable differences between SA2 regions within Sydney - Parramatta. Areas such as North Parramatta, Ermington - Rydalmere, and Merrylands - Holroyd achieved relatively high scores due to larger POI concentrations. In contrast, areas such as South Wentworthville, Auburn - North, and Auburn - South received lower scores because they contained fewer recorded POIs.

These outputs were later used to generate the visualisations and comparative analysis presented in the report.

In [5]:
display(load_score_input_summary(engine, settings))

scores = load_sa2_scores(engine, settings)
score_map_areas = load_sa2_scores(engine, settings, include_excluded=True)
display(scores.head())
display(build_top_bottom_table(scores, n=settings.charts.top_n))

,sa2_count,total_poi,mean_poi_count,std_poi_count,min_poi_count,max_poi_count,below_min_population,missing_population
0,34,2086,61.352941,34.151808,3,132,3,0


,sa2_code,sa2_name,sa4_code,sa4_name,population,poi_count,mean_poi_count,std_poi_count,z_poi,score_raw,score_100,is_excluded,exclusion_reason,geometry
0,125011582,Auburn - Central,125,Sydney - Parramatta,19751,68,66.548387,31.179852,0.046556,0.511637,51.163693,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
1,125011583,Auburn - North,125,Sydney - Parramatta,10580,21,66.548387,31.179852,-1.460828,0.188341,18.834078,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
2,125011584,Auburn - South,125,Sydney - Parramatta,8953,22,66.548387,31.179852,-1.428756,0.193293,19.329265,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
3,125011585,Berala,125,Sydney - Parramatta,8502,20,66.548387,31.179852,-1.492900,0.183487,18.348692,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
4,125021711,Carlingford - East,125,Sydney - Parramatta,11586,48,66.548387,31.179852,-0.594884,0.355515,35.551508,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."


,rank_group,sa2_code,sa2_name,sa4_name,poi_count,score_100,population
0,top,125041489,North Parramatta,Sydney - Parramatta,132,89.082187,22919
1,top,125021477,Ermington - Rydalmere,Sydney - Parramatta,118,83.891211,25193
2,top,125031714,Merrylands - Holroyd,Sydney - Parramatta,112,81.118430,24083
3,top,125031481,Granville - Clyde,Sydney - Parramatta,105,77.438163,23627
4,top,125041491,Northmead,Sydney - Parramatta,97,72.644179,20401
5,top,125031484,Guildford West - Merrylands West,Sydney - Parramatta,96,72.002224,23285
6,top,125041717,Parramatta - North,Sydney - Parramatta,94,70.691090,12467
7,top,125011710,Wentworth Point - Sydney Olympic Park,Sydney - Parramatta,94,70.691090,21985
8,top,125031483,Guildford - South Granville,Sydney - Parramatta,84,63.638498,22542
9,top,125011586,Lidcombe,Sydney - Parramatta,83,62.893158,21483


## Individual Visual Analysis
This section prepares the datasets used for the final visual analysis. The workflow loads POI group counts, POI coordinate data, and score-income comparison data from the database before generating maps and charts.

These datasets are later used to explore the spatial distribution of POIs, compare well-resourced scores between SA2 regions, and examine possible relationships between accessibility scores and median income levels.


In [6]:
poi_groups = load_poi_group_counts(engine, settings)
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit)
score_income = load_score_income(engine, settings)

### Score Distribution
This histogram shows the distribution of well-resourced scores across the selected SA2 regions. Most regions were concentrated within the middle score range, while only a smaller number of areas achieved very high or very low scores.

The distribution suggests that accessibility and resource concentration were not evenly balanced across the selected SA4 region. Several SA2 areas achieved moderate scores, while a few regions with stronger POI concentrations appeared as higher-scoring outliers. This pattern indicates noticeable variation in how urban resources and facilities are distributed between different areas.

In [7]:
build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show()

### Top and Bottom SA2 Scores
These charts compare the highest and lowest scoring SA2 regions within the selected SA4 area. Clear differences can be observed between regions with high POI concentrations and areas with fewer recorded facilities and services.

North Parramatta achieved the highest score in the selected region, followed by areas such as Ermington - Rydalmere and Merrylands - Holroyd. These regions generally contained larger numbers of POIs and stronger concentrations of transport, recreation, and community facilities. In contrast, lower-scoring areas such as South Wentworthville, Auburn - North, Auburn - South, and Parramatta - South recorded noticeably smaller POI counts and lower accessibility scores. Many of these regions appeared to be more residential suburban areas with fewer public-facing services and facilities included in the dataset.

The comparison between the top and bottom ranking groups suggests that accessibility and urban resources are not distributed evenly across the SA4 region. Some neighbouring areas showed large differences in score values despite being located within the same broader region, indicating that local infrastructure concentration and urban development patterns may strongly influence the final well-resourced scores.

In [8]:
build_top_sa2_bar(scores, n=settings.charts.top_n).show()
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show()


### Score Choropleth Map
This choropleth map visualises the distribution of well-resourced scores across SA2 regions within Sydney - Parramatta. Clear spatial differences can be observed between areas with higher accessibility scores and regions with lower levels of recorded services and facilities.

Several central and northern parts of the region achieved relatively high scores, shown by the lighter yellow and green areas on the map. These regions generally contained higher POI counts and stronger concentrations of transport, recreation, and community-related facilities. In contrast, darker blue and purple areas represented lower-scoring regions with fewer recorded POIs and lower overall accessibility levels.

The map also highlights that neighbouring SA2 regions can have noticeably different scores despite being geographically close to each other. This suggests that local infrastructure development and service concentration may vary significantly within the broader Parramatta area.

In [9]:
build_score_choropleth_map(score_map_areas).show()

### Population-Adjusted POI Density Map
This map shows the population-adjusted POI density across SA2 regions within Sydney - Parramatta. Instead of only comparing total POI counts, the visualisation considers the number of POIs relative to population size, allowing differences in accessibility intensity between regions to be compared more fairly.

Several smaller or more commercially concentrated areas achieved relatively high POI density values, shown by the brighter orange and yellow colours on the map. These regions contained larger numbers of facilities and services relative to their resident population. In contrast, darker purple areas had lower POI density values, suggesting that facilities and services were more limited when adjusted for population size.

Compared with the previous score map, the population-adjusted density map provides a slightly different perspective on accessibility. Some areas with moderate total POI counts still achieved relatively high density values because of smaller populations, while some highly populated suburban areas appeared less resource-dense after population adjustment.

In [10]:
build_poi_density_choropleth_map(score_map_areas).show()


### POI Point Map
This map visualises the spatial distribution of POIs across the selected SA4 region. Different colours represent different POI categories, including transport, recreation, education, community services, and other facility types.

The map shows that POIs were generally concentrated around more urbanised and commercially active areas, while outer suburban regions contained fewer recorded facilities. Recreation, transport, and community-related POIs appeared most frequently across the selected region, suggesting that these categories contributed strongly to the final well-resourced scores.

The clustering of points in several central areas also indicates that accessibility and services were unevenly distributed across the region. Some SA2 areas contained dense concentrations of facilities within relatively small geographic spaces, while other surrounding regions appeared more sparsely serviced.

In [11]:
build_poi_point_scatter_map(poi_points).show()

### POI Group Distribution
This chart shows the distribution of POI categories across the selected SA4 region. Recreation-related POIs formed the largest group in the dataset by a significant margin, followed by community and transport facilities. In comparison, categories such as utility, hydrography, and landform appeared much less frequently.

The results suggest that recreational and community-focused facilities were the most common types of recorded services within the region. These categories likely contributed strongly to the final well-resourced scores because of their large presence across many SA2 areas.

The uneven distribution between POI categories also highlights a limitation of the dataset and scoring approach. Some categories naturally contain more records than others, which may influence the final accessibility scores more heavily than less common facility types.

In [12]:
build_poi_group_distribution(poi_groups).show()

### Score and Median Income
This scatter plot compares well-resourced scores with median income levels across SA2 regions within Sydney - Parramatta. A weak positive relationship can be observed, as areas with higher median incomes generally tended to achieve slightly higher accessibility scores.

However, the relationship was not particularly strong, and there was still considerable variation between regions with similar income levels. Some middle-income areas achieved relatively high scores because of stronger concentrations of transport, recreation, and community facilities, while several higher-income regions still recorded only moderate accessibility scores.

The results suggest that income alone does not fully explain differences in accessibility and resource distribution across the region. Local infrastructure, land use patterns, and service concentration may also play important roles in influencing the final well-resourced scores.

In [13]:
build_score_income_scatter(score_income).show()

## Correlation and Interpretation Notes
This section summarises the statistical relationship between median income and well-resourced scores within the selected SA4 region. Both Pearson and Spearman correlation tests produced weak positive correlation values, suggesting that higher-income areas may sometimes be associated with slightly higher accessibility scores.

However, the p-values for both tests were above the selected significance threshold of 0.05, meaning that the relationship was not statistically significant in this dataset. This indicates that income alone was not a strong predictor of well-resourced scores across the analysed SA2 regions.

The results support the earlier visual analysis from the scatter plot, where some higher-income regions achieved only moderate scores, while several middle-income areas still performed relatively well because of stronger concentrations of local services and facilities.

In [14]:
display(load_correlation_summary(engine, settings))

,method,statistic,p_value,n,alpha,is_significant,created_at,interpretation
0,pearson,0.197362,0.287228,31,0.05,False,2026-05-20 13:00:02.197945+00:00,not statistically significant
1,spearman,0.226347,0.220793,31,0.05,False,2026-05-20 13:00:02.197945+00:00,not statistically significant


## Key Findings
The analysis shows that well-resourced scores were not evenly distributed across Greater Sydney. Most high-scoring SA2 regions were concentrated around highly urbanised and economically active areas, while lower-scoring regions were more common in outer suburban or primarily residential locations. The spatial distribution on the score map also suggests that accessibility and service concentration are strongly connected to urban density and development intensity.

Several regions in Sydney - City and Inner South achieved extremely high scores, with some SA2 areas scoring close to 100. These locations also recorded very high POI counts, indicating strong concentrations of transport services, recreation facilities, commercial activity, and community infrastructure. Northern Beaches and North Sydney and Hornsby also contained multiple high-scoring regions, although their score distribution appeared more spread out across neighbouring SA2 areas rather than concentrated in only one centre.

Compared with these regions, Sydney - Parramatta showed a more mixed pattern. North Parramatta achieved a relatively high score and appeared in the top-ranking group, suggesting that some parts of the region have strong accessibility and urban activity. However, several nearby SA2 areas such as Auburn - North, Auburn - South, Parramatta - South, and South Wentworthville were included in the bottom ranking group. This contrast indicates that resource distribution within Parramatta itself is uneven, with some areas benefiting from stronger infrastructure and service concentration while others remain less well-resourced.

The histogram of well-resourced scores further supports this observation. Most SA2 areas were concentrated within the middle score range, while only a small number achieved very high scores above 80. This suggests that extremely well-resourced areas are relatively limited across the metropolitan region. At the same time, the score distribution does not appear completely polarised, since many areas still achieved moderate accessibility levels rather than extremely low scores.

The relationship between median income and well-resourced score appeared slightly positive overall, but the trend was not particularly strong. Higher income areas often achieved better scores, although several middle-income regions also recorded relatively high accessibility values. This suggests that income alone does not fully determine how well-resourced an area is. Urban structure, transport accessibility, land use, and facility concentration are also likely to influence the final score.

The POI distribution also provides useful context for understanding the scoring results. Recreation, community, and transport related POIs formed the largest proportion of the dataset, while industry and utility related POIs appeared much less frequently. This means the scoring system was influenced more heavily by public-facing urban services and lifestyle facilities than by industrial infrastructure. Areas with larger concentrations of these everyday facilities generally achieved higher overall scores.

One limitation of the analysis is that the scoring system mainly depends on POI counts and simplified weighting methods. The model does not fully consider service quality, travel time, transport efficiency, or differences in population demand between regions. Therefore, the well-resourced score should be interpreted as a general indicator of accessibility and service concentration rather than a complete measurement of urban liveability or infrastructure quality.